# Project: The Data Analyst Agent

**What you will build:** a capstone that combines everything — an agent that answers questions about a dataset by **writing and running its own Python** (a code-execution tool), with **memory**, a **guardrail**, and a small **eval**. It's the code version of the n8n "Ask Your Data" project.

One general "run code" tool beats a hundred narrow ones — the same idea behind coding agents like Claude Code.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/32_project_data_analyst.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(
    base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"]))
print("Ready:", MODEL_NAME)
import pandas as pd, io, contextlib

## The data

A tiny sales dataset. In a real project this would be your CSV, a database query, or an uploaded file.

In [ ]:
CSV = """month,product,units,revenue
Jan,Widget,120,2400
Jan,Gadget,80,3200
Feb,Widget,150,3000
Feb,Gadget,95,3800
Mar,Widget,90,1800
Mar,Gadget,130,5200"""
df = pd.read_csv(io.StringIO(CSV))
df

## The code-execution tool

Instead of writing one tool per question (`total_revenue()`, `best_month()`, ...), we give the agent a single tool that runs Python against the dataframe. The agent writes the analysis itself. This is *programmatic tool use*: one general "run code" tool instead of a hundred narrow ones.

In [ ]:
def run_python(code: str) -> str:
    """Run Python to analyze the sales dataframe `df` (pandas is `pd`). print() the answer."""
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {"df": df, "pd": pd})
        return buf.getvalue().strip() or "(ran, but nothing was printed — remember to print() the result)"
    except Exception as e:
        return f"Error: {e}"   # feed the error back so the agent can fix its own code (0.3, 19)

print(run_python("print(df['revenue'].sum())"))   # quick sanity check

```{warning}
`exec` runs arbitrary code — fine for *your own* analysis in a notebook, but **never expose this to untrusted users as-is**. In production you'd sandbox it, in rough order of strength: a locked-down `subprocess` with a timeout, a container, or a hosted code-interpreter service (e.g. E2B). We keep it simple here to teach the pattern; the n8n course makes the same trade-off and says so.
```

```{note}
**Each call starts fresh — there is no memory between tool calls.** `run_python` passes a brand-new namespace (`{"df": df, "pd": pd}`) every time, so a variable the agent defines in one call is gone by the next. The agent must compute each answer in a *single self-contained block*. A hosted code-interpreter (Jupyter, E2B) keeps state across cells like a REPL; this deliberately doesn't, which makes every tool call reproducible in isolation — but it's a real constraint the agent's system prompt should imply ("print the result in one block").
```

## The agent (the system prompt *is* the engineering)

The workflow is three lines; the quality lives in the system prompt. We tell the agent the schema, that it must use the tool, and to answer only from what the code prints — the same iterative prompt-engineering spine as n8n's Ask Your Data.

In [ ]:
SCHEMA = "The dataframe `df` has columns: month (Jan/Feb/Mar), product (Widget/Gadget), units (int), revenue (USD int)."

analyst = Agent(
    model,
    system_prompt=(
        "You are a data analyst. Answer questions about the sales data by writing Python and calling run_python. "
        f"{SCHEMA} Always print() the result inside the code. Base your answer ONLY on what the tool prints, "
        "and state the numbers you found."
    ),
)
analyst.tool_plain(run_python)   # register the function defined above as a tool

r = await analyst.run("Which product made more total revenue, and by how much?")
print(r.output)

The agent wrote pandas code, ran it, read the printed numbers, and answered from them. Now let's layer on the rest of the course.

## Production upgrade: a human gate before code runs

That `exec` warning wasn't rhetorical. The moment anyone *other than you* can trigger a code-execution tool, put a human in front of it. In a notebook the simplest gate is `input()`; in a durable service it's the `interrupt()` pattern from 2.4 — the graph pauses, persists its state, and resumes only after a reviewer approves. This is HITL (0.5, 2.4) applied exactly where it earns its cost: a write/act tool (1.5).

In [ ]:
def run_python_reviewed(code: str) -> str:
    """run_python, but a human must approve the code before it executes."""
    print("The agent wants to run:\n")
    print(code)
    if input("\nType RUN to approve: ").strip() != "RUN":
        return "Cancelled by the human reviewer — nothing was executed."
    return run_python(code)

# Try it — you'll be prompted before anything runs:
print(run_python_reviewed("print(df['revenue'].sum())"))

Register `run_python_reviewed` instead of `run_python` and every generated snippet waits for a human "RUN" before touching `exec`. A human in *every* loop doesn't scale, of course — so in practice you gate by **risk rating** (1.5): high-risk tools (running code, writing to a DB, spending money) get the gate; read-only lookups run free. The point is that the approval lives in your code, not in a hopeful line of the system prompt.

## Add memory: follow-up questions

Pass `message_history` (1.4) so the agent handles follow-ups that depend on the previous turn.

In [ ]:
r1 = await analyst.run("What was total revenue in February?")
print("Q1:", r1.output)

r2 = await analyst.run("And how does that compare to January?", message_history=r1.all_messages())
print("Q2:", r2.output)   # 'that' resolves via memory

## Add a guardrail: stay on task

A data agent shouldn't answer off-topic questions (or run unrelated code). A quick input screen (1.5) keeps it scoped.

In [ ]:
async def ask_analyst(question, history=None):
    off_topic = ["weather", "joke", "who are you", "password"]
    if any(w in question.lower() for w in off_topic):
        return "I only answer questions about the sales dataset."
    r = await analyst.run(question, message_history=history or [])
    return r.output

print(await ask_analyst("Tell me a joke"))                      # refused
print(await ask_analyst("Which month had the highest revenue?"))  # answered

## Add an eval: is it actually right? (and a trap in the grader)

Non-deterministic + writing its own code = you must **measure** (1.6). We compute ground truth with pandas and check the agent's answer. But *how* you check is itself a design decision that can silently lie to you — so we'll grade the way 1.6 warned you to, not the naive way.

The naive grader is `truth.lower() in answer.lower()` — is the right value a substring of the reply? It looks fine until a **comparison** question breaks it: for "which product sold more units?" the true answer is `Widget`, but a *wrong* reply — "Gadget sold more than Widget" — still contains the string "widget", so the substring test scores it **PASS**. A green eval that certifies a wrong answer is worse than no eval. The fix is to make the agent *commit* to a parseable final answer and grade only that:

In [ ]:
import re

def committed_answer(text: str) -> str:
    """The agent's final, parseable answer: the last 'ANSWER:' line (falls back to the last line)."""
    hits = re.findall(r"ANSWER:\s*(.+)", text, flags=re.IGNORECASE)
    return (hits[-1] if hits else text.strip().splitlines()[-1]).strip()

def norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", s.lower())          # drop $, commas, spaces, case

def grade(truth: str, got: str) -> bool:
    # Grade the COMMITTED value, not the prose — exact for categories, digit-subset for numbers.
    if norm(got) == norm(truth):
        return True
    return truth.isdigit() and truth in re.sub(r"[^0-9]", "", got)   # "18400" inside "18400 USD"

cases = [
    ("What is the total revenue across all months?", str(df['revenue'].sum())),
    ("How many Widget units were sold in total?",     str(df[df['product']=='Widget']['units'].sum())),
    ("Which product sold more total units?",
     "Widget" if df[df.product=='Widget'].units.sum() > df[df.product=='Gadget'].units.sum() else "Gadget"),
]

passed = 0
for question, truth in cases:
    # Force a committed, bare answer — this is what defuses the comparison trap.
    reply = await ask_analyst(question + "\nEnd with exactly one line: 'ANSWER: <the value only>'.")
    got = committed_answer(reply)
    ok = grade(truth, got)
    passed += ok
    print(f"[{'PASS' if ok else 'FAIL'}] want {truth!r:>8}  got ANSWER={got[:32]!r}")
print(f"\nScore: {passed}/{len(cases)}")

# The trap, demonstrated: the naive grader would PASS a wrong comparison answer.
naive_pass = "widget" in "Gadget sold more than Widget".lower()
print(f"\nnaive substring grader on a WRONG answer -> {'PASS (!!)' if naive_pass else 'FAIL'}  <- why we grade the committed value")

That score is your ground truth. Change the model or the system prompt, re-run this cell, and you *know* whether it got better — the whole point of evals. And note the last line: the naive substring grader scores a **wrong** comparison answer as PASS, which is why we graded the agent's *committed* `ANSWER:` value instead. The transferable rule: **an eval is only as trustworthy as its grader — make the agent commit to a checkable answer, and grade that, not the surrounding prose** (1.6's lesson, learned the hard way).

**You combined tools + memory + guardrails + evals into one agent.** That is enough for a portfolio project.

::::{dropdown} 🛠️ Extend it (challenges)
:color: secondary

1. **Use your own data.** Replace `CSV` with a real dataset (load a file in Colab) and update `SCHEMA`. Watch how much the schema in the prompt matters.
2. **Add a chart tool.** Give the agent a second tool that runs matplotlib and saves a PNG; ask it to plot revenue by month.
3. **Harden the guardrail.** Move the off-topic check into an `@analyst.output_validator` (1.5) and also block code that imports modules.
4. **Grow the eval.** Add five more cases with known answers and track your pass rate as you tweak the prompt.
5. **Ship it.** Wrap this agent in the FastAPI service from 30 and call it from a front end (31).
::::

---

**Project complete.** You combined tools, memory, guardrails, and evals into one agent. In **32b**, you will build a different shape of system: typed triage and deterministic routing for employee onboarding.